# AST (Audio Spectrogram Transformer) — Alternative to PANN


## 1. Setup

In [ ]:
import numpy as np
import torch
import librosa

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

from transformers import ASTFeatureExtractor, ASTModel

from kitty3000_ml.preprocess import load_clip
from kitty3000_ml.labels import LABELS                     # MK: import the label set for classification from central source

In [ ]:
DATA_DIR = "../data/raw/CatSound_originals"                   # MK: unzip the originals file from  gdrive into the data directory
MANIFEST = "../data/manifest.csv"                             # MK: or ../data/manifest_dirty.csv

SAMPLE_RATE = 16000                                           # AST's expected input rate
SECONDS = 11.0                                                # MK: team decision 09.09., based on Ceven's duration tests (2/5/7/11 s)

CLASS_NAMES = [l for l in LABELS if l != "Unknown"]   # 10 classes, same order as the API (labels.py); Unknown is V2

MODEL_NAME = "ast_audioset_frozen_logreg"
AST_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"

## 2. Load dataset


In [ ]:
manifest = pd.read_csv(MANIFEST)

def load_split(split):
    rows = manifest[manifest["split"] == split]
    waveforms = []
    labels = []
    for i in rows.index:
        path = DATA_DIR + "/" + rows["path"][i]
        waveforms.append(load_clip(path, SAMPLE_RATE, SECONDS))
        labels.append(CLASS_NAMES.index(rows["label"][i]))     # "Defence" -> 1, same order as labels.py
    return waveforms, labels

train_waveforms, train_labels = load_split("train")
test_waveforms, test_labels = load_split("val")     # val for now - test stays untouched

print(f"\nLoaded {len(train_waveforms) + len(test_waveforms)} clips across {len(CLASS_NAMES)} classes")

In [ ]:
print(f"Train: {len(train_waveforms)} | Val: {len(test_waveforms)}")

## 3. Load AST and extract embeddings


In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

feature_extractor = ASTFeatureExtractor.from_pretrained(AST_CHECKPOINT)
ast_model = ASTModel.from_pretrained(AST_CHECKPOINT).to(device)
ast_model.eval()


def extract_ast_embedding(waveform, sr=SAMPLE_RATE):
    """Returns a single fixed-length embedding vector for one clip (mean-pooled over patches)."""
    inputs = feature_extractor(waveform, sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = ast_model(**inputs)

    last_hidden = outputs.last_hidden_state
    pooled = last_hidden.mean(dim=1).squeeze(0)
    return pooled.cpu().numpy()

In [ ]:
print("Extracting embeddings for train set...")
X_train = np.stack([extract_ast_embedding(w) for w in train_waveforms])

print("Extracting embeddings for test set...")
X_test = np.stack([extract_ast_embedding(w) for w in test_waveforms])

print(X_train.shape, X_test.shape)

## 4. Train classifier head


In [ ]:
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train, train_labels)

## 5. Evaluate

In [ ]:
def ast_predict_fn(waveform, sr):
    embedding = extract_ast_embedding(waveform, sr)
    return clf.predict(embedding[None, :])[0]


y_true = test_labels
y_pred = clf.predict(X_test)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

## 6. Compare against PANN

(PANN, AST, and any future models)

In [ ]:
#Not yet ready

## 7. Duration-bias sanity check

In [ ]:
val_rows = manifest[manifest["split"] == "val"]
durations = [librosa.get_duration(path=DATA_DIR + "/" + p) for p in val_rows["path"]]
misclassified = [
    (round(d, 1), CLASS_NAMES[t], CLASS_NAMES[p])
    for d, t, p in zip(durations, y_true, y_pred)
    if t != p
]
print(f"{len(misclassified)} misclassified clips — inspect for duration patterns:")
misclassified[:10]